In [1]:
import sys, os
from typing import Optional
import numpy as np
import pandas as pd
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/manushshah/Desktop/FRE 9743 - Models to Markets/GitHub/FRE-GT-9743-Assignment-5
Fixed Income Library is loaded.


# Instructions for HW5:

## PCA implementation

### 1. Write PCA algorithm

Please implement a PCA algorithm without using existing PCA api, which means you're only allowed to use Numpy package.

In [2]:
def pca(data: np.ndarray, n_components: Optional[int] = None, use_corr: bool = False):
    # centering
    mean = np.mean(data, axis=0)
    X = data - mean

    # standardization
    if use_corr:
        std = np.std(X, axis=0, ddof=1)
        std[std == 0] = 1.0
        X = X / std

    # Covariance matrix
    cov_matrix = np.cov(X, rowvar=False, ddof=1)

    # Eigen values and vectors
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    total_var = eigenvalues.sum()

    if n_components is not None:
        eigenvalues = eigenvalues[:n_components]
        eigenvectors = eigenvectors[:, :n_components]

    scores = X @ eigenvectors

    explained_var = eigenvalues / total_var

    return eigenvalues, eigenvectors, scores, explained_var

### 2. Apply PCA

Please download fixed-floating swap rate data from Bloomberg (Ticker: YCSW0490) covering at least six months, with the sample period ending on 2026-02-26.

Although you should download the data for 2026-02-27, exclude that date from the PCA analysis.

The swap tenors should range from 1Y to 30Y.

The dataset should be organized in the format shown below.

After preparing the dataset, apply Principal Component Analysis (PCA) as discussed in class.

![Chart](data_sample.png)

In [3]:
# ── Load and clean swap data ──
raw = pd.read_excel("data.xlsx", header=0)
raw.rename(columns={"Dates": "Date", "12M": "1Y"}, inplace=True)
raw.drop(columns=["18M"], inplace=True)

# Parse dates and sort chronologically
raw["Date"] = pd.to_datetime(raw["Date"])
raw.sort_values("Date", inplace=True)
raw.reset_index(drop=True, inplace=True)

# Tenor columns
tenors = ["1Y","2Y","3Y","4Y","5Y","6Y","7Y","8Y","9Y","10Y","12Y","15Y","20Y","25Y","30Y"]
raw[tenors] = raw[tenors].astype(float)

print(f"Data: {raw['Date'].iloc[0].date()} to {raw['Date'].iloc[-1].date()}, {len(raw)} rows")
print(raw.head())



Data: 2025-08-30 to 2026-02-26, 130 rows
        Date      1Y      2Y      3Y      4Y      5Y      6Y      7Y      8Y  \
0 2025-08-30  3.7515  3.3908  3.2940  3.2945  3.3350  3.4003  3.4749  3.5497   
1 2025-09-01  3.7630  3.3908  3.2940  3.2945  3.3350  3.4003  3.4749  3.5497   
2 2025-09-02  3.7601  3.4141  3.3193  3.3170  3.3560  3.4218  3.4980  3.5733   
3 2025-09-03  3.7344  3.3910  3.2965  3.2937  3.3311  3.3936  3.4662  3.5388   
4 2025-09-04  3.7020  3.3546  3.2565  3.2493  3.2829  3.3433  3.4141  3.4865   

       9Y     10Y     12Y     15Y     20Y     25Y     30Y  
0  3.6217  3.6910  3.8185  3.9622  4.0832  4.1065  4.0844  
1  3.6217  3.6910  3.8185  3.9622  4.0832  4.1065  4.0844  
2  3.6463  3.7165  3.8448  3.9903  4.1122  4.1359  4.1139  
3  3.6097  3.6778  3.8022  3.9428  4.0593  4.0799  4.0553  
4  3.5570  3.6253  3.7506  3.8939  4.0143  4.0375  4.0151  


In [4]:
# ── Daily changes (in bps) ──
df_levels = raw.set_index("Date")[tenors]
df_changes = df_levels.diff().dropna() * 100  # convert pct to bps

print(f"\nDaily changes: {len(df_changes)} observations x {len(tenors)} tenors")
print(df_changes.head())


Daily changes: 129 observations x 15 tenors
              1Y    2Y    3Y    4Y    5Y    6Y    7Y    8Y    9Y   10Y   12Y  \
Date                                                                           
2025-09-01  1.15  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00   
2025-09-02 -0.29  2.33  2.53  2.25  2.10  2.15  2.31  2.36  2.46  2.55  2.63   
2025-09-03 -2.57 -2.31 -2.28 -2.33 -2.49 -2.82 -3.18 -3.45 -3.66 -3.87 -4.26   
2025-09-04 -3.24 -3.64 -4.00 -4.44 -4.82 -5.03 -5.21 -5.23 -5.27 -5.25 -5.16   
2025-09-05 -9.00 -7.57 -6.42 -6.23 -6.34 -6.77 -7.28 -7.57 -7.80 -8.04 -8.32   

             15Y   20Y   25Y   30Y  
Date                                
2025-09-01  0.00  0.00  0.00  0.00  
2025-09-02  2.81  2.90  2.94  2.95  
2025-09-03 -4.75 -5.29 -5.60 -5.86  
2025-09-04 -4.89 -4.50 -4.24 -4.02  
2025-09-05 -8.49 -8.54 -8.58 -8.60  


In [5]:
# ── Run PCA on daily changes ──
eigenvalues, eigenvectors, scores, explained_var = pca(df_changes.values, n_components=3)

print("\n── PCA Results ──")
for i in range(3):
    print(f"PC{i+1}: eigenvalue = {eigenvalues[i]:.6f}, explained variance = {explained_var[i]:.4%}")
print(f"Total explained by 3 PCs: {explained_var.sum():.4%}")

# ── Display loadings ──
loadings_df = pd.DataFrame(eigenvectors, index=tenors, columns=["PC1","PC2","PC3"])
print("\nPC Loadings:")
print(loadings_df.round(4))


── PCA Results ──
PC1: eigenvalue = 159.060555, explained variance = 93.3322%
PC2: eigenvalue = 8.993564, explained variance = 5.2772%
PC3: eigenvalue = 1.943896, explained variance = 1.1406%
Total explained by 3 PCs: 99.7500%

PC Loadings:
        PC1     PC2     PC3
1Y   0.1927  0.3750  0.7093
2Y   0.2426  0.4041  0.2909
3Y   0.2650  0.3293 -0.0880
4Y   0.2724  0.2553 -0.2071
5Y   0.2781  0.1763 -0.2686
6Y   0.2783  0.1025 -0.2457
7Y   0.2779  0.0322 -0.2002
8Y   0.2747 -0.0214 -0.1587
9Y   0.2726 -0.0682 -0.1187
10Y  0.2693 -0.1096 -0.0769
12Y  0.2631 -0.1770 -0.0056
15Y  0.2549 -0.2463  0.0754
20Y  0.2453 -0.3098  0.1569
25Y  0.2385 -0.3552  0.2162
30Y  0.2324 -0.3855  0.2584


### 3. Hedging with PCA

With your PCA results, please try to reconstruct the risk profile provided, and present your results.

In [6]:
# read risk_agg.csv
risk_agg = pd.read_csv(os.path.join(repo_root,"risk_agg.csv"))
print("Risk Aggregation Data:")
risk_agg.head()

Risk Aggregation Data:


,Unnamed: 0,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,12Y,15Y,20Y,25Y,30Y
0,AGG_RISK,2.982972,-1.857554,0.013279,0.071185,13.975129,-5.458591,-0.025772,-7.003277,0.043624,12.783767,-15.256859,-21.13873,-17.444482,0.0,0.0


In [7]:
# ── Extract numeric risk vector ──
tenors = ["1Y","2Y","3Y","4Y","5Y","6Y","7Y","8Y","9Y","10Y","12Y","15Y","20Y","25Y","30Y"]
risk_vec = risk_agg[tenors].values.flatten().astype(float)  # (15,)

# ── Project risk into PC space ──
# eigenvectors from Part 2: shape (15, 3), columns are PC1, PC2, PC3
risk_pc = eigenvectors.T @ risk_vec  # (3,)

print("Risk profile in PC space:")
print(f"  PC1 (shift):     {risk_pc[0]:.4f}")
print(f"  PC2 (slope):     {risk_pc[1]:.4f}")
print(f"  PC3 (curvature): {risk_pc[2]:.4f}")

# ── Reconstruct risk from top 3 PCs ──
risk_reconstructed = eigenvectors @ risk_pc  # (15,)
residual = risk_vec - risk_reconstructed

print("\nReconstruction comparison:")
recon_df = pd.DataFrame({
    "Original": risk_vec,
    "Reconstructed (3 PCs)": risk_reconstructed,
    "Residual": residual
}, index=tenors)
print(recon_df.round(4))
print(f"\nResidual norm: {np.linalg.norm(residual):.6f}")
print(f"Reconstruction R²: {1 - np.sum(residual**2)/np.sum(risk_vec**2):.6%}")

Risk profile in PC space:
  PC1 (shift):     -9.6425
  PC2 (slope):     14.3503
  PC3 (curvature): -4.9693

Reconstruction comparison:
     Original  Reconstructed (3 PCs)  Residual
1Y     2.9830                -0.0015    2.9845
2Y    -1.8576                 2.0134   -3.8709
3Y     0.0133                 2.6085   -2.5952
4Y     0.0712                 2.0659   -1.9947
5Y    13.9751                 1.1821   12.7930
6Y    -5.4586                 0.0083   -5.4668
7Y    -0.0258                -1.2219    1.1961
8Y    -7.0033                -2.1682   -4.8350
9Y     0.0436                -3.0180    3.0616
10Y   12.7838                -3.7872   16.5709
12Y  -15.2569                -5.0480  -10.2089
15Y  -21.1387                -6.3668  -14.7719
20Y  -17.4445                -7.5908   -9.8537
25Y    0.0000                -8.4714    8.4714
30Y    0.0000                -9.0571    9.0571

Residual norm: 33.320450
Reconstruction R²: 22.568715%


Using the provided hedging instruments, solve for the optimal portfolio weights that hedge the aggregated DV01 risk profile. 

Compare the eigenspace approach against direct least squares, and argue mathematically why the former is preferred.

In [8]:
hedging = pd.read_csv(os.path.join(repo_root,"hedging_instruments.csv"))
print("Hedging Instruments Data:")
hedging.head()

Hedging Instruments Data:


,Unnamed: 0,1Y,2Y,3Y,4Y,5Y,6Y,7Y,8Y,9Y,10Y,12Y,15Y,20Y,25Y,30Y
0,1Y1Y,0.990383,-1.905780,-0.040437,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,1Y5Y,0.986415,0.006578,0.000573,0.001714,0.049129,-5.468511,-0.035436,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
2,5Y5Y,0.002970,0.005982,0.009149,0.012336,4.626952,-0.016352,-0.019360,-0.022309,-0.025584,-8.501197,0.000000,0.000000,0.000000,0.0,0.0
3,10Y10Y,0.003699,0.007465,0.011399,0.015366,0.019642,0.023980,0.028543,0.032885,0.037736,8.499983,-0.093768,-0.198309,-14.335121,0.0,0.0


In [10]:
# ── Extract hedging instrument matrix ──
tenors = ["1Y","2Y","3Y","4Y","5Y","6Y","7Y","8Y","9Y","10Y","12Y","15Y","20Y","25Y","30Y"]
risk_vec = risk_agg[tenors].values.flatten().astype(float)   # (15,)
hedge_mat = hedging[tenors].values.astype(float)              # (4, 15)
instruments = hedging.iloc[:, 0].values                       # ['1Y1Y', '1Y5Y', '5Y5Y', '10Y10Y']

# ── Method 1: Eigenspace Hedge (project into PC space, solve there) ──
risk_pc = eigenvectors.T @ risk_vec          # (3,)
hedge_pc = (eigenvectors.T @ hedge_mat.T).T  # (4, 3)

# Least squares in 3-dimensional PC space: hedge_pc @ w = risk_pc
weights_eigen, _, _, _ = np.linalg.lstsq(hedge_pc.T, risk_pc, rcond=None)

print("── Method 1: Eigenspace Hedge (3 PCs) ──")
for name, w in zip(instruments, weights_eigen):
    print(f"  {name}: {w:.6f}")

hedged_eigen = risk_vec - weights_eigen @ hedge_mat
print(f"  Residual DV01 norm: {np.linalg.norm(hedged_eigen):.6f}")

# ── Method 2: Direct Least Squares (fit all 15 tenors) ──
# hedge_mat.T is (15, 4), risk_vec is (15,) → solve for w (4,)
weights_direct, _, _, _ = np.linalg.lstsq(hedge_mat.T, risk_vec, rcond=None)

print("\n── Method 2: Direct Least Squares ──")
for name, w in zip(instruments, weights_direct):
    print(f"  {name}: {w:.6f}")

hedged_direct = risk_vec - weights_direct @ hedge_mat
print(f"  Residual DV01 norm: {np.linalg.norm(hedged_direct):.6f}")

# ── Side-by-side comparison ──
print("\n── Residual DV01 by Tenor ──")
comp_df = pd.DataFrame({
    "Original": risk_vec,
    "Eigenspace Residual": hedged_eigen,
    "Direct LS Residual": hedged_direct
}, index=tenors)
print(comp_df.round(4))

print(f"\n── Summary ──")
print(f"Eigenspace residual norm:  {np.linalg.norm(hedged_eigen):.4f}")
print(f"Direct LS residual norm:   {np.linalg.norm(hedged_direct):.4f}")
print(f"\nDirect LS has smaller total residual, but eigenspace hedging")
print(f"concentrates residual in dimensions with minimal curve variance.")

── Method 1: Eigenspace Hedge (3 PCs) ──
  1Y1Y: 0.326547
  1Y5Y: 2.556915
  5Y5Y: 2.331090
  10Y10Y: 3.021648
  Residual DV01 norm: 38.674626

── Method 2: Direct Least Squares ──
  1Y1Y: 1.193592
  1Y5Y: 1.045629
  5Y5Y: 0.674989
  10Y10Y: 1.487148
  Residual DV01 norm: 29.646868

── Residual DV01 by Tenor ──
     Original  Eigenspace Residual  Direct LS Residual
1Y     2.9830               0.1193              0.7619
2Y    -1.8576              -1.2885              0.3952
3Y     0.0133              -0.0308              0.0378
4Y     0.0712              -0.0084              0.0382
5Y    13.9751               3.0043             10.7714
6Y    -5.4586               8.4896              0.2348
7Y    -0.0258               0.0237             -0.0181
8Y    -7.0033              -7.0506             -7.0371
9Y     0.0436              -0.0108              0.0048
10Y   12.7838               6.9169              5.8812
12Y  -15.2569             -14.9735            -15.1174
15Y  -21.1387             -

why Eigenspace Hedging is Preferred ──

1. NOISE FILTERING
   
   PCA decomposes curve movements into orthogonal modes ranked by variance. With 3 PCs explaining 99.75%, the remaining 12 dimensions are noise.
   Direct LS wastes hedge capacity fitting noise dimensions. Eigenspace hedging targets only the 3 modes where real P&L risk lives.

2. BETTER CONDITIONING
   Direct LS solves: H'w = r  (15 eqns, 4 unknowns → overdetermined)
   Eigenspace solves: V'H'w = V'r  (3 eqns, 4 unknowns)
   
   In tenor space, H' has condition number inflated by near-zero variance directions. Projecting via V_k regularizes the problem — the system
   is solved only in directions with meaningful signal.

3. MATHEMATICAL FORMULATION
   Let V = [v1, v2, v3] be the top 3 eigenvectors.
   
   Direct LS:     min_w  ||r - H'w||²  =  Σ over all 15 tenors
   Eigenspace:    min_w  ||V'(r - H'w)||²  =  Σ over 3 PCs only
   
   Direct LS minimizes: Σ_i (residual in tenor i)²
   Eigenspace minimizes: Σ_k λ_k-weighted (residual in PC k)²
   
   The eigenspace objective is variance-weighted — a 1 unit residual in
   PC1 (λ=159) matters 80x more than in PC12 (λ≈0.02). Direct LS treats
   them equally, leading to overfitting in irrelevant dimensions.

4. ECONOMIC INTERPRETATION
   PC1 = parallel shift, PC2 = slope, PC3 = curvature.
   Eigenspace hedging directly neutralizes exposure to the 3 dominant
   risk factors a rates trader actually manages. Residual risk lives
   in higher-order modes where historical curve variance is negligible —
   economically, this residual is diversified away.